In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("/kaggle/input/notebooks/madisrinija/02-feature-engineering-ipynb/student_performance_dataset_cleaned.csv")
df.shape

(1000, 19)

In [3]:
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade,at_risk,academic_level,attendance_risk_level,sleep_deviation,deviation_bin,high_workload_low_sleep,engagement_score
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A,0,excellent,Healthy,1.0,1.0,0,0.805426
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A,0,excellent,Healthy,1.8,2.0,1,0.921053
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A,0,excellent,Healthy,0.4,0.0,0,0.417909
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B,0,excellent,Healthy,0.5,0.0,0,0.592843
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D,1,average,Healthy,2.9,3.0,1,0.331199


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   object 
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    object 
 6   internet_access             1000 non-null   object 
 7   extracurricular_activities  1000 non-null   object 
 8   part_time_job               1000 non-null   object 
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   object 
 12  at_risk                     1000 non-null   int64  
 13  academic_level              1000 n

In [5]:
results = pd.DataFrame(columns=[
    'Model', 'Recall', 'Precision', 'F1 Score', 'ROC-AUC'
])

In [6]:
X = df.drop(columns='final_grade')
y = df['final_grade']

In [7]:
cat_cols = X.select_dtypes(include='object').columns
print(cat_cols)

Index(['gender', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'academic_level',
       'attendance_risk_level'],
      dtype='object')


In [8]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


In [9]:
X_train,X_test,Y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 1. Logistic Regression

In [10]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('logreg', LogisticRegression())
])

model.fit(X_train, Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'academic_level',
       'attendance_risk_level'],
      dtype='object'))])),
                ('logreg', LogisticRegression())])

In [11]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [12]:
results.loc[len(results)] = [
    'Logistic Regression',
    recall_score(y_test, y_pred, average='weighted'),
    precision_score(y_test, y_pred, average='weighted'),
    f1_score(y_test, y_pred, average='weighted'),
    roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
]

## 2. Random Forest

In [13]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('rfc', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'academic_level',
       'attendance_risk_level'],
      dtype='object'))])),
                ('rfc', RandomForestClassifier(random_state=42))])

In [14]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [15]:
results.loc[len(results)] = [
    'Random Forest Classifier',
    recall_score(y_test, y_pred, average='weighted'),
    precision_score(y_test, y_pred, average='weighted'),
    f1_score(y_test, y_pred, average='weighted'),
    roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
]

## 3. XGBoost Classifier

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(Y_train)
y_test_encoded = le.fit_transform(y_test)

In [17]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42))
])

model.fit(X_train, y_train_encoded)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'parental_education', 'internet_access',
       'extracurricular_activities', 'part_time_job', 'academic_level',
       'attendance_risk_level'],
      dtype='object'))])),
                ('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_byl...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [18]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [19]:
results.loc[len(results)] = [
    'XGBoost',
    recall_score(y_test_encoded, y_pred, average='weighted'),
    precision_score(y_test_encoded, y_pred, average='weighted'),
    f1_score(y_test_encoded, y_pred, average='weighted'),
    roc_auc_score(
        y_test_encoded,
        y_prob,
        multi_class='ovr',
        average='weighted'
    )
]

In [20]:
results

,Model,Recall,Precision,F1 Score,ROC-AUC
0,Logistic Regression,0.620,0.602092,0.574771,0.841746
1,Random Forest Classifier,0.620,0.614148,0.606299,0.843553
2,XGBoost,0.625,0.624552,0.608659,0.844114
